# Hands-On Machine Learning with PyTorch (Aurelion Geron)

## 1) Building Neural Networks with PyTorch

### 2. Sequential Models

#### Linear Regression

In [9]:
# Let us first implement linear regression using the sklearn california housing dataset 

import torch

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()

X_train, X_test, y_train, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

print(f"Initially, X_train, X_test, and X_val all have type {type(X_train)}.")

X_train = torch.FloatTensor(X_train)
X_val = torch.FloatTensor(X_val)
X_test = torch.FloatTensor(X_test)

mean = X_train.mean(dim=0, keepdims=True)
std = X_train.std(dim=0, keepdims=True)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std 
X_test = (X_test - mean) / std 

y_train = torch.FloatTensor(y_train).reshape(-1,1)
y_val = torch.FloatTensor(y_val).reshape(-1,1)
y_test = torch.FloatTensor(y_test).reshape(-1,1)

torch.manual_seed(42)

n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

lr = 0.4
n_epochs = 20

for epoch in range(n_epochs):
    y_pred = X_train @ w + b 
    loss = ((y_pred - y_train)**2).mean()
    loss.backward()

    with torch.no_grad():
        b -= lr * b.grad 
        w -= lr * w.grad 
    
    b.grad.zero_()
    w.grad.zero_()

    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Initially, X_train, X_test, and X_val all have type <class 'numpy.ndarray'>.
Epoch 1/20, Loss: 15.992842674255371
Epoch 2/20, Loss: 4.682830333709717
Epoch 3/20, Loss: 2.1424787044525146
Epoch 4/20, Loss: 1.2740927934646606
Epoch 5/20, Loss: 0.9412086009979248
Epoch 6/20, Loss: 0.8018259406089783
Epoch 7/20, Loss: 0.735872209072113
Epoch 8/20, Loss: 0.699141800403595
Epoch 9/20, Loss: 0.6748808026313782
Epoch 10/20, Loss: 0.6565617322921753
Epoch 11/20, Loss: 0.6415442228317261
Epoch 12/20, Loss: 0.6286912560462952
Epoch 13/20, Loss: 0.617459774017334
Epoch 14/20, Loss: 0.6075484156608582
Epoch 15/20, Loss: 0.5987609028816223
Epoch 16/20, Loss: 0.5909508466720581
Epoch 17/20, Loss: 0.5839995741844177
Epoch 18/20, Loss: 0.5778065919876099
Epoch 19/20, Loss: 0.5722847580909729
Epoch 20/20, Loss: 0.5673576593399048


In [11]:
# We can directly use PyTorch's high-level API

import torch.nn as nn 

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)   # Linear layer with n_features many inputs and 1 output feature 

print(f"The weights of the linear model are {model.weight} with bias {model.bias}.\n")


for i, param in enumerate(model.parameters()):
    print(f"Param {i} is {param}.\n")

for n, val in model.named_parameters():
    print(f"Parameter {n} has value {val}.\n")

# We can call a module like a regular function 

print(f"The model predicts {model(X_train[:2])} for the first 2 rows.")


# Choose optimizer, here SGD
lr = 0.4
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
mse = nn.MSELoss()
n_epochs = 20

def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train) 
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item()}.")

train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

The weights of the linear model are Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True) with bias Parameter containing:
tensor([0.3117], requires_grad=True).

Param 0 is Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True).

Param 1 is Parameter containing:
tensor([0.3117], requires_grad=True).

Parameter weight has value Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True).

Parameter bias has value Parameter containing:
tensor([0.3117], requires_grad=True).

The model predicts tensor([[0.6524],
        [0.5043]], grad_fn=<AddmmBackward0>) for the first 2 rows.
Epoch 1/20, Loss: 4.27860164642334.
Epoch 2/20, Loss: 0.7788285613059998.
Epoch 3/20, Loss: 0.6250892877578735.
Epoch 4/20, Loss: 0.605811595916748.
Epoch 5/20, Loss: 0.595306932926178.
Epoch 6

#### Regression MLP

In [22]:
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

lr = 0.1
n_epochs = 20
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
mse = nn.MSELoss()

train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.990477561950684.
Epoch 2/20, Loss: 2.0748581886291504.
Epoch 3/20, Loss: 1.026237964630127.
Epoch 4/20, Loss: 0.8820208311080933.
Epoch 5/20, Loss: 0.7992834448814392.
Epoch 6/20, Loss: 0.7477124333381653.
Epoch 7/20, Loss: 0.7145673632621765.
Epoch 8/20, Loss: 0.6924373507499695.
Epoch 9/20, Loss: 0.6769211292266846.
Epoch 10/20, Loss: 0.6653458476066589.
Epoch 11/20, Loss: 0.6560705304145813.
Epoch 12/20, Loss: 0.6481837034225464.
Epoch 13/20, Loss: 0.6411526203155518.
Epoch 14/20, Loss: 0.6347273588180542.
Epoch 15/20, Loss: 0.6286888122558594.
Epoch 16/20, Loss: 0.6229467988014221.
Epoch 17/20, Loss: 0.6174619793891907.
Epoch 18/20, Loss: 0.6121846437454224.
Epoch 19/20, Loss: 0.6070876121520996.
Epoch 20/20, Loss: 0.6021547317504883.


In [34]:
# We can use Mini-Batch GD using DataLoaders

from torch.utils.data import TensorDataset, DataLoader 

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          pin_memory=True, num_workers=2, prefetch_factor=2,
                          persistent_workers=True)

torch.manual_seed(42) 

model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu" 

model = model.to(device)

n_epochs = 20
lr = 0.1 
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
criterion = nn.MSELoss()

def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()

    for epoch in range(n_epochs):
        total_loss = 0

        # In each epoch, we sample batches from train_loader, and these batches are 
        # used, where the dataloader shuffles the dataset in each epoch and samples  

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)

            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        print(f"total_loss of epoch {epoch +1} is {total_loss}.\n")
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}.\n\n")

train(model, optimizer, criterion, train_loader, n_epochs)

total_loss of epoch 1 is 241.0169532597065.

Epoch 1/20, Loss: 0.5528.


total_loss of epoch 2 is 189.58528795838356.

Epoch 2/20, Loss: 0.4348.


total_loss of epoch 3 is 175.7169238179922.

Epoch 3/20, Loss: 0.4030.


total_loss of epoch 4 is 171.67939821630716.

Epoch 4/20, Loss: 0.3938.


total_loss of epoch 5 is 161.0794636234641.

Epoch 5/20, Loss: 0.3694.


total_loss of epoch 6 is 155.59230738133192.

Epoch 6/20, Loss: 0.3569.


total_loss of epoch 7 is 149.3167555257678.

Epoch 7/20, Loss: 0.3425.


total_loss of epoch 8 is 146.32639841735363.

Epoch 8/20, Loss: 0.3356.


total_loss of epoch 9 is 143.94635467231274.

Epoch 9/20, Loss: 0.3302.


total_loss of epoch 10 is 141.12818975001574.

Epoch 10/20, Loss: 0.3237.


total_loss of epoch 11 is 141.21311827749014.

Epoch 11/20, Loss: 0.3239.


total_loss of epoch 12 is 138.77595326304436.

Epoch 12/20, Loss: 0.3183.


total_loss of epoch 13 is 137.30249556154013.

Epoch 13/20, Loss: 0.3149.


total_loss of epoch 14 is 135.8850

In [ ]:
print(f"model is currently in {'training' if model.training else 'inference'}.")

# Now let us write evaluation method

def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)

    return aggregate_fn(torch.stack(metrics))

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True,
                        pin_memory=True, num_workers=2, prefetch_factor=2,
                        persistent_workers=True)

val_mse = evaluate(model, val_loader, criterion)
print(f"MSE on validation set is {val_mse}") 

model is currently in training.


MSE on test set is 0.34493228793144226
